# Práctica MLOps - Versión esencial

Notebook resumido de la práctica de clasificación de latidos ECG con MIT-BIH. Se conserva solo lo imprescindible: preparación de datos, modelos evaluados y comparación de resultados.

## 1. Objetivo

Comparar varios enfoques de clasificación multiclase sobre la señal ECG:

- Baseline `DummyClassifier`
- Regresión logística multinomial
- Random Forest sobre *features* manuales
- MLP
- CNN 1D

La métrica principal para comparar modelos será `macro_f1`, porque el dataset está desbalanceado.

In [ ]:
# Configuración e imports
SEED = 42
FAST_RUN = False

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

In [ ]:
# Carga de datos
base_dir = Path.cwd()
candidate_dirs = [
    base_dir,
    base_dir / "MLOPS" / "prcatica",
    base_dir.parent,
]

def find_file(filename):
    for d in candidate_dirs:
        p = d / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"No se encuentra {filename}")

train_path = find_file("mitbih_train.csv")
test_path = find_file("mitbih_test.csv")

train_df = pd.read_csv(train_path, header=None)
test_df = pd.read_csv(test_path, header=None)

feature_cols = [f"t_{i}" for i in range(train_df.shape[1] - 1)]
columns = feature_cols + ["label"]
train_df.columns = columns
test_df.columns = columns

print("Shape train:", train_df.shape)
print("Shape test :", test_df.shape)

In [ ]:
# Preparación de datos
label_map = {
    0: "N (normal)",
    1: "S (supraventricular)",
    2: "V (ventricular)",
    3: "F (fusionado)",
    4: "Q (desconocido)",
}

X_train_full = train_df[feature_cols].values.astype(np.float32)
y_train_full = train_df["label"].values.astype(int)
X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df["label"].values.astype(int)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_full,
)

scaler_raw = StandardScaler()
X_train_scaled = scaler_raw.fit_transform(X_train)
X_val_scaled = scaler_raw.transform(X_val)
X_test_scaled = scaler_raw.transform(X_test)

X_train_cnn = X_train[..., np.newaxis]
X_val_cnn = X_val[..., np.newaxis]
X_test_cnn = X_test[..., np.newaxis]

classes = np.unique(y_train)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, class_weights)}

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)
print("Class weights:", class_weight_dict)

In [ ]:
# Funciones auxiliares
results = []

def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

def register_result(model_name, split_name, y_true, y_pred):
    row = {"modelo": model_name, "split": split_name}
    row.update(compute_metrics(y_true, y_pred))
    results.append(row)
    return row

def extract_time_features(X):
    X = np.asarray(X, dtype=np.float32)
    return pd.DataFrame({
        "mean": X.mean(axis=1),
        "std": X.std(axis=1),
        "min": X.min(axis=1),
        "max": X.max(axis=1),
        "median": np.median(X, axis=1),
        "q25": np.quantile(X, 0.25, axis=1),
        "q75": np.quantile(X, 0.75, axis=1),
        "ptp": np.ptp(X, axis=1),
        "energy": np.sum(X**2, axis=1),
        "mean_abs": np.mean(np.abs(X), axis=1),
        "argmax": np.argmax(X, axis=1),
        "argmin": np.argmin(X, axis=1),
        "signal_length": np.sum(np.abs(np.diff(X, axis=1)), axis=1),
        "zero_crossings": np.sum(np.diff(np.signbit(X), axis=1), axis=1),
        "skew": skew(X, axis=1, bias=False, nan_policy="omit"),
        "kurtosis": kurtosis(X, axis=1, fisher=True, bias=False, nan_policy="omit"),
    }).replace([np.inf, -np.inf], np.nan).fillna(0.0)

## 2. Modelos evaluados

In [ ]:
# Baseline: clase mayoritaria
dummy_clf = DummyClassifier(strategy="most_frequent")
dummy_clf.fit(X_train, y_train)
y_val_pred_dummy = dummy_clf.predict(X_val)
register_result("Dummy - clase mayoritaria", "val", y_val, y_val_pred_dummy)

In [ ]:
# Regresión logística multinomial
logreg = LogisticRegression(
    max_iter=500,
    multi_class="multinomial",
    solver="lbfgs",
    class_weight="balanced",
    random_state=SEED,
)
logreg.fit(X_train_scaled, y_train)
y_val_pred_logreg = logreg.predict(X_val_scaled)
register_result("LogReg multinomial", "val", y_val, y_val_pred_logreg)

In [ ]:
# Random Forest sobre features manuales
X_train_feat = extract_time_features(X_train)
X_val_feat = extract_time_features(X_val)
X_test_feat = extract_time_features(X_test)

rf = RandomForestClassifier(
    n_estimators=300 if not FAST_RUN else 120,
    min_samples_leaf=2,
    n_jobs=-1,
    class_weight="balanced_subsample",
    random_state=SEED,
)
rf.fit(X_train_feat, y_train)
y_val_pred_rf = rf.predict(X_val_feat)
register_result("RandomForest + features", "val", y_val, y_val_pred_rf)

In [ ]:
# MLP
BATCH_SIZE = 256 if not FAST_RUN else 128
EPOCHS = 30 if not FAST_RUN else 8

def build_mlp(input_dim, n_classes=5):
    model = models.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.30),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.25),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.20),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

mlp = build_mlp(X_train_scaled.shape[1], len(label_map))
early_stop = callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_mlp = mlp.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    verbose=0,
    callbacks=[early_stop],
)

y_val_pred_mlp = np.argmax(mlp.predict(X_val_scaled, verbose=0), axis=1)
register_result("MLP", "val", y_val, y_val_pred_mlp)

In [ ]:
# CNN 1D
def build_cnn1d(input_shape, n_classes=5):
    inp = layers.Input(shape=input_shape)
    x = layers.Conv1D(32, kernel_size=7, padding="same", activation="relu")(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(64, kernel_size=5, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, kernel_size=3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)

    model = models.Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

cnn = build_cnn1d((X_train_cnn.shape[1], 1), len(label_map))
early_stop_cnn = callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)

history_cnn = cnn.fit(
    X_train_cnn,
    y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    verbose=0,
    callbacks=[early_stop_cnn],
)

y_val_pred_cnn = np.argmax(cnn.predict(X_val_cnn, verbose=0), axis=1)
register_result("CNN 1D", "val", y_val, y_val_pred_cnn)

In [ ]:
# Comparativa en validación
val_results = (
    pd.DataFrame(results)
    .query("split == 'val'")
    .sort_values(["macro_f1", "balanced_accuracy", "accuracy"], ascending=False)
    .reset_index(drop=True)
)
val_results.round(4)

## 3. Resultados observados en el notebook original

| Modelo | Accuracy (val) | Balanced Accuracy (val) | Macro F1 (val) |
|---|---:|---:|---:|
| RandomForest + features | 0.9653 | 0.8026 | 0.8468 |
| MLP | 0.9038 | 0.9279 | 0.7159 |
| CNN 1D | 0.9070 | 0.8914 | 0.7142 |
| LogReg multinomial | 0.6776 | 0.7787 | 0.4857 |
| Dummy - clase mayoritaria | 0.8277 | 0.2000 | 0.1811 |

Resultado en test del mejor modelo seleccionado (`RandomForest + features`):

| Modelo | Accuracy (test) | Balanced Accuracy (test) | Macro F1 (test) |
|---|---:|---:|---:|
| RandomForest + features | 0.9636 | 0.7905 | 0.8368 |

## 4. Conclusión

Aunque `MLP` y `CNN 1D` mejoran claramente el tratamiento del desbalance si miramos `balanced_accuracy`, el mejor compromiso global en esta práctica lo obtuvo `RandomForest + features`, que alcanzó el mayor `macro_f1` en validación y mantuvo un rendimiento muy alto en test.

En otras palabras, para este problema concreto, una buena ingeniería de características temporales con un modelo clásico ha superado a las redes neuronales profundas del notebook original.